# Exploratory Data Analysis – Phishing URL Detection

This notebook explores the **Phishing Dataset for Machine Learning** (Kaggle, *shashwatwork*).

We answer:
1. What is the class distribution?
2. Are there missing values / duplicates?
3. Which features are most correlated with the target?
4. How do key features differ between phishing and legitimate URLs?

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

from src.preprocessing import load_raw_dataset, clean_dataframe
from src.feature_engineering import engineer_features, get_feature_column_names
from src.utils import load_config

## 1. Load the data

In [ ]:
df = load_raw_dataset()
df = clean_dataframe(df)
df = engineer_features(df)
df.head()

In [ ]:
print('Shape :', df.shape)
print('Missing values :', df.isna().sum().sum())
print('Duplicated rows :', df.duplicated().sum())

## 2. Class distribution

In [ ]:
lbl_col = load_config()['columns']['label_column']
ax = sns.countplot(data=df, x=lbl_col, palette='Set2')
ax.set_xticklabels(['Legitimate (0)', 'Phishing (1)'])
ax.set_title('Class distribution')
plt.savefig('../results/class_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Feature distributions

In [ ]:
top_features = ['UrlLength', 'NumDots', 'NumDash', 'HostnameLength', 'PathLength', 'NumNumericChars']
fig, axes = plt.subplots(2, 3, figsize=(14, 8))
for ax, feat in zip(axes.flat, top_features):
    sns.boxplot(data=df, x=lbl_col, y=feat, ax=ax, palette='Set2')
    ax.set_title(feat)
    ax.set_xticklabels(['Legit', 'Phish'])
plt.tight_layout()
plt.savefig('../results/feature_distributions.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Correlation heatmap

In [ ]:
feature_cols = get_feature_column_names()
corr = df[feature_cols + [lbl_col]].corr()
plt.figure(figsize=(16, 14))
sns.heatmap(corr, cmap='coolwarm', center=0, square=True, cbar_kws={'shrink': 0.7})
plt.title('Feature correlation heatmap')
plt.tight_layout()
plt.savefig('../results/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

### Top features correlated with the target

In [ ]:
target_corr = corr[lbl_col].drop(lbl_col).sort_values(key=lambda s: s.abs(), ascending=False)
target_corr.head(15)

## 5. Missing-values visual check

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
plt.figure(figsize=(10, 4))
sns.barplot(x=missing.index[:15], y=missing.values[:15], palette='Reds_r')
plt.xticks(rotation=45, ha='right')
plt.title('Missing values per column (top 15)')
plt.tight_layout()
plt.savefig('../results/missing_values.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. PCA 2-D projection

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

X = df[feature_cols].values
y = df[lbl_col].values
X_std = StandardScaler().fit_transform(X)
X_2d = PCA(n_components=2, random_state=42).fit_transform(X_std)

plt.figure(figsize=(8, 6))
sns.scatterplot(x=X_2d[:, 0], y=X_2d[:, 1], hue=y, palette='Set1', alpha=0.4, s=12)
plt.title('PCA projection of feature space')
plt.xlabel('PC1'); plt.ylabel('PC2')
plt.legend(title='Class', labels=['Legitimate', 'Phishing'])
plt.tight_layout()
plt.savefig('../results/pca_projection.png', dpi=150, bbox_inches='tight')
plt.show()

### Key EDA findings
* The dataset is **perfectly balanced** (5000 / 5000), so accuracy is a meaningful first metric.
* There are **no missing values** and no duplicates.
* Features most correlated with phishing include `PctExtHyperlinks`, `PctNullSelfRedirectHyperlinks`, and the engineered `DigitToUrlRatio`.
* PCA shows that a single linear projection is **not** sufficient to separate the classes, which justifies using a non-linear tree-based model.